In [1]:
import numpy as np
import urllib.request
import gzip
import os

def load_images(path):
    with gzip.open(path, 'rb') as f:
        data = np.frombuffer(f.read(), np.uint8, offset=16)
    return data.reshape(-1, 28, 28)

def load_labels(path):
    with gzip.open(path, 'rb') as f:
        return np.frombuffer(f.read(), np.uint8, offset=8)

In [2]:
base_url = "http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/"

files = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz"
}

for file in files.values():
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

x_train = load_images(files["train_images"])
y_train = load_labels(files["train_labels"])
x_test = load_images(files["test_images"])
y_test = load_labels(files["test_labels"])

x_train = x_train.reshape(x_train.shape[0], -1) / 255.0
x_test = x_test.reshape(x_test.shape[0], -1) / 255.0

x_train = x_train[:6000]
y_train = y_train[:6000]
x_test = x_test[:1000]
y_test = y_test[:1000]

def euclidean(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def knn_predict(x_train, y_train, x, k):
    distances = []
    for i in range(len(x_train)):
        distances.append((euclidean(x_train[i], x), y_train[i]))
    distances.sort(key=lambda z: z[0])
    labels = [label for _, label in distances[:k]]
    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]

In [3]:
predictions = []
misclassified = []

k = 5

for i in range(len(x_test)):
    pred = knn_predict(x_train, y_train, x_test[i], k)
    predictions.append(pred)
    if pred != y_test[i]:
        misclassified.append(i)

predictions = np.array(predictions)
accuracy = np.sum(predictions == y_test) / len(y_test)

print("Accuracy:", accuracy)
print("Misclassified samples:", len(misclassified))

Accuracy: 0.812
Misclassified samples: 188
